## 4.11 SQLite 데이터베이스 생성 및 연결

In [ ]:
import sqlite3

conn = sqlite3.connect("test.db")
cursor = conn.cursor()

## 4.12 SQLite 테이블 생성

In [3]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS logs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
    user_input TEXT,
    model_output TEXT
)
''')
conn.commit()

## 4.13 SQLite 데이터 삽입

In [4]:
cursor.execute('''
INSERT INTO logs (user_input, model_output)
VALUES (?, ?)
''', ("Example input", "Example output"))
conn.commit()

## 4.13 SQLite 데이터 조회

In [5]:
cursor.execute('SELECT * FROM logs')
rows = cursor.fetchall()
print(rows)

## 4.14 SQLite 연결 종료

In [7]:
conn.close()

## 4.47 완전 일치 문자열 평가자를 이용한 평가 예시

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from llmops_lib.evaluator import create_evaluator, EvaluatorType

# 문장 감정 분류 체인 정의 
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "주어진 문장의 감정을 긍정, 부정, 중립으로만 분류합니다. 분류에 대한 추가 설명을 하지 않습니다."), 
    ("user", "문장: {sentence}\n분류:")
])
llm = ChatOllama(model="mistral", temperature=0.1, max_tokens=100)
chain = prompt_template | llm

# 입력 변수
input_variables = {"sentence": "오늘 날씨 참 좋다"}
# 참조 출력값
reference_output = "긍정"

# Exact Match 평가자 생성
evaluator = create_evaluator(EvaluatorType.EXACT_MATCH, chain=chain)

# 평가 실행
evaluator.evaluate(input_variables=input_variables, reference_output=reference_output)

{'input_variables': {'sentence': '오늘 날씨 참 좋다'},
 'output': '긍정',
 'reference_output': '긍정',
 'input_token': 86,
 'output_token': 5,
 'latency': 7.486641883850098,
 'score': 1.0}